<div dir="rtl">
<h1>خروجی سالم، مسیر یادگیری قطع‌شده</h1>
<p>درس 52 از 76 · Gradient چگونه تا جدول Embedding می‌رسد؟ · <code dir="ltr">46-gradient-path</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/46-gradient-path.html">📖 بازگشت به همین درس</a></p>
<p>None و Gradient صفر را تشخیص دهید و محل قطع مسیر پیش از Head را پیدا کنید.</p><p>پیش‌نیاز: backward، zero_grad، detach و Norm را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>با detachکردن hidden پیش از Head، کدام Parameterها هنوز Gradient می‌گیرند؟ آیا Shape خروجی باید تغییر کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
model = MiniGPT(ModelConfig(12,8,8,2,2,0.)).eval()
ids = torch.tensor([[1,2,3,4]])
targets = torch.tensor([[2,3,4,5]])
def features(model, ids):
    positions = torch.arange(ids.shape[1])
    hidden = model.dropout(model.token_embedding(ids)+model.position_embedding(positions))
    for block in model.blocks:
        hidden = block(hidden)
    return model.final_norm(hidden)
print("tracked endpoints: token_embedding.weight / language_model_head.weight")

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع gradient_report(Model) دیکشنری نام Parameter به None یا float اندازهٔ Gradient برگرداند. None را به صفر تبدیل نکنید. خود تابع forward یا backward اجرا نکند و مشتق‌ها را تغییر ندهد.</p>
</div>

In [ ]:
def gradient_report(model):
    # TODO
    return None

In [ ]:
def test_exercise():
    model.zero_grad(set_to_none=True)
    result = gradient_report(model)
    if result is None: return False
    assert set(result) == {name for name,_ in model.named_parameters()}
    assert all(value is None for value in result.values())
    model(ids,targets)[1].backward()
    report = gradient_report(model)
    assert all(value is not None and math.isfinite(value) for value in report.values())
    for name,p in model.named_parameters():
        assert abs(report[name]-p.grad.norm().item()) < 1e-8
    model.zero_grad(set_to_none=True)
    (model(ids,targets)[1]*0).backward()
    assert all(value == 0. for value in gradient_report(model).values())
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>این بار از forward کامل مدل استفاده کنید و فقط ضریب Loss را از یک به صفر تغییر دهید؛ وزن و Batch ثابت باشند. پیش از هر backward مشتق‌ها را با set_to_none=True پاک کنید. وقتی Gradient صفر می‌شود، آیا هنوز Tensor موجودی داریم یا مقدار آن None شده است؟</p>
</div>

In [ ]:
for loss_scale in (1.,0.):
    model.zero_grad(set_to_none=True)
    _,loss = model(ids,targets)
    (loss_scale*loss).backward()
    embedding_gradient = model.token_embedding.weight.grad
    print('loss scale, gradient missing, nonzero entries:',loss_scale,embedding_gradient is None,torch.count_nonzero(embedding_gradient).item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب مقدار hidden را درست نگه می‌دارد ولی ارتباط آن با Layer‌های قبلی را قطع می‌کند. تابع connected_head_loss(Model,hidden,targets) را بدون detach اصلاح کنید؛ Loss Tensor برگردانید، نه item.</p>
</div>

In [ ]:
model.zero_grad(set_to_none=True)
hidden = features(model,ids)
wrong_logits = model.language_model_head(hidden.detach())
F.cross_entropy(wrong_logits.reshape(-1,12),targets.reshape(-1)).backward()
print('valid logits shape:',wrong_logits.shape,'embedding grad:',model.token_embedding.weight.grad)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def connected_head_loss(model, hidden, targets):
    # TODO
    return None

In [ ]:
def test_repair():
    model.zero_grad(set_to_none=True)
    hidden = features(model,ids)
    result = connected_head_loss(model,hidden,targets)
    if result is None: return False
    torch.testing.assert_close(result,model(ids,targets)[1])
    result.backward()
    assert model.token_embedding.weight.grad is not None
    assert model.language_model_head.weight.grad is not None
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    model.zero_grad(set_to_none=True)
    other_ids,other_targets = ids[:,:2],targets[:,:2]
    loss = connected_head_loss(model,features(model,other_ids),other_targets)
    loss.backward()
    assert model.blocks[0].attention.qkv.weight.grad is not None
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>test_all_parameters_receive_gradients در tests/test_model.py مسیر سالم مدل را می‌سنجد. این دفتر علاوه بر آن، همان وزن‌ها را با یک قطع عمدی آزمایش کرد تا معنای هر Gradient موجود یا غایب روشن شود.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر پس از صفرکردن Loss مشتق صفر دارید ولی پس از detach مشتق None، این دو مشاهده دربارهٔ ساختار مسیر چه تفاوتی دارند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/46-gradient-path.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/46-gradient-path.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>